# San Francisco Employee Salaries (2011–2018)
## Data Processing, Error Handling & File Management Assignment

**Dataset:** [Kaggle – SF Salaries](https://www.kaggle.com/datasets/mojtaba142/20112018-salaries-for-san-francisco?select=Total.csv)  
**Columns:** `Id`, `EmployeeName`, `JobTitle`, `BasePay`, `OvertimePay`, `OtherPay`, `Benefits`, `TotalPay`, `TotalPayBenefits`, `Year`, `Notes`, `Agency`, `Status`


---
## Task 1 – Import Data

In [ ]:
import pandas as pd
import numpy as np
import csv
import zipfile
import os

# ── Load the dataset ──────────────────────────────────────────────────────────
# Place Total.csv in the same directory as this notebook, OR update the path.
DATA_FILE = "Total.csv"

try:
    df = pd.read_csv(DATA_FILE)
    print(f"✅ Dataset loaded successfully: {df.shape[0]:,} rows × {df.shape[1]} columns")
except FileNotFoundError:
    raise FileNotFoundError(
        f"'{DATA_FILE}' not found. Download Total.csv from Kaggle and place it "
        "in the same folder as this notebook."
    )

df.head()

In [ ]:
# Basic overview
print("Columns:", df.columns.tolist())
print()
df.info()

In [ ]:
# ── Light cleaning ────────────────────────────────────────────────────────────
# Normalise numeric pay columns (some entries may be strings like 'Not Provided')
PAY_COLS = ["BasePay", "OvertimePay", "OtherPay", "Benefits", "TotalPay", "TotalPayBenefits"]

for col in PAY_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Strip whitespace from name column
df["EmployeeName"] = df["EmployeeName"].astype(str).str.strip()

print("Null counts after cleaning:")
print(df[PAY_COLS].isnull().sum())

---
## Task 2 – Employee Look-up Function

In [ ]:
def get_employee_details(name: str, dataframe: pd.DataFrame = df) -> pd.DataFrame:
    """
    Return all records for an employee whose name contains `name` (case-insensitive).

    Parameters
    ----------
    name : str
        Full or partial employee name to search for.
    dataframe : pd.DataFrame
        The salary DataFrame to search within (defaults to the global `df`).

    Returns
    -------
    pd.DataFrame
        Rows matching the name query.

    Raises
    ------
    TypeError
        If `name` is not a string.
    ValueError
        If no employee is found with the given name.
    """
    # ── Input validation ──────────────────────────────────────────────────────
    if not isinstance(name, str):
        raise TypeError(f"Employee name must be a string, got {type(name).__name__!r}.")

    name = name.strip()
    if not name:
        raise ValueError("Employee name cannot be empty.")

    # ── Search ────────────────────────────────────────────────────────────────
    mask = dataframe["EmployeeName"].str.contains(name, case=False, na=False)
    result = dataframe[mask].copy()

    if result.empty:
        raise ValueError(f"No employee found matching '{name}'.")

    return result


# ── Demo ──────────────────────────────────────────────────────────────────────
sample_name = df["EmployeeName"].dropna().iloc[0]   # use the very first employee as demo
print(f"Looking up: '{sample_name}'")
get_employee_details(sample_name)

---
## Task 3 – Data Processing with a Dictionary

In [ ]:
# ── Build a nested dictionary: { EmployeeName -> { year -> record_dict } } ────
def build_salary_dict(dataframe: pd.DataFrame) -> dict:
    """
    Convert the salary DataFrame into a nested dictionary.

    Structure
    ---------
    {
      'EMPLOYEE NAME': {
          year_int: { column: value, ... },
          ...
      },
      ...
    }
    """
    salary_dict: dict = {}

    for _, row in dataframe.iterrows():
        emp_name: str = row["EmployeeName"]
        year: int = int(row["Year"]) if pd.notna(row["Year"]) else 0

        if emp_name not in salary_dict:
            salary_dict[emp_name] = {}

        salary_dict[emp_name][year] = row.to_dict()

    return salary_dict


salary_dict = build_salary_dict(df)
print(f"✅ Dictionary built with {len(salary_dict):,} unique employees.")

# Show one sample entry
first_key = next(iter(salary_dict))
print(f"\nSample entry for '{first_key}':")
for yr, details in salary_dict[first_key].items():
    print(f"  Year {yr}: TotalPayBenefits = {details.get('TotalPayBenefits')}")

In [ ]:
# ── Summary statistics derived from the dictionary ────────────────────────────
def summarise_employee(name: str, salary_dict: dict) -> dict:
    """
    Return a summary dict of pay statistics for an employee across all years.
    """
    if name not in salary_dict:
        raise KeyError(f"'{name}' not found in salary dictionary.")

    records = salary_dict[name]
    total_pays = [
        v["TotalPayBenefits"]
        for v in records.values()
        if pd.notna(v.get("TotalPayBenefits"))
    ]

    return {
        "employee": name,
        "years_on_record": sorted(records.keys()),
        "avg_total_pay_benefits": round(np.mean(total_pays), 2) if total_pays else None,
        "max_total_pay_benefits": round(max(total_pays), 2)  if total_pays else None,
        "min_total_pay_benefits": round(min(total_pays), 2)  if total_pays else None,
    }


summary = summarise_employee(first_key, salary_dict)
print("Employee Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

---
## Task 4 – Error Handling

In [ ]:
# ── Demonstrate robust error handling across all helper functions ──────────────

test_cases = [
    ("Valid name",          lambda: get_employee_details(first_key)),
    ("Empty string",        lambda: get_employee_details("")),
    ("Non-string input",    lambda: get_employee_details(12345)),
    ("Name not in dataset", lambda: get_employee_details("ZZZZZZ_NONEXISTENT_ZZZZZZ")),
    ("Dict key missing",    lambda: summarise_employee("GHOST EMPLOYEE", salary_dict)),
]

for label, fn in test_cases:
    try:
        result = fn()
        if isinstance(result, pd.DataFrame):
            print(f"[{label}] ✅ Returned {len(result)} row(s).")
        else:
            print(f"[{label}] ✅ {result}")
    except (TypeError, ValueError, KeyError) as exc:
        print(f"[{label}] ⚠️  {type(exc).__name__}: {exc}")
    except Exception as exc:
        print(f"[{label}] ❌ Unexpected {type(exc).__name__}: {exc}")

---
## Task 5 – Export Employee Details to CSV → ZIP

In [ ]:
def export_employee_profile(
    name: str,
    dataframe: pd.DataFrame = df,
    output_dir: str = ".",
) -> str:
    """
    Export an employee's records to a CSV file, then zip it into
    'Employee Profile/<name>.zip'.

    Returns the path to the created ZIP file.
    """
    # ── Retrieve records (raises on bad input) ────────────────────────────────
    records = get_employee_details(name, dataframe)

    # ── Sanitise name for use as filename ─────────────────────────────────────
    safe_name = "".join(c if c.isalnum() or c in " _-" else "_" for c in name).strip()

    # ── Paths ─────────────────────────────────────────────────────────────────
    profile_dir = os.path.join(output_dir, "Employee Profile")
    os.makedirs(profile_dir, exist_ok=True)

    csv_filename  = f"{safe_name}.csv"
    csv_path      = os.path.join(profile_dir, csv_filename)
    zip_path      = os.path.join(profile_dir, f"{safe_name}.zip")

    # ── Write CSV ─────────────────────────────────────────────────────────────
    try:
        records.to_csv(csv_path, index=False)
        print(f"✅ CSV written: {csv_path}  ({len(records)} row(s))")
    except IOError as exc:
        raise IOError(f"Failed to write CSV: {exc}") from exc

    # ── Zip the CSV ───────────────────────────────────────────────────────────
    try:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(csv_path, arcname=csv_filename)
        print(f"✅ ZIP created: {zip_path}")
    except Exception as exc:
        raise RuntimeError(f"Failed to create ZIP: {exc}") from exc

    return zip_path


# ── Export the first employee as a demo ──────────────────────────────────────
zip_file = export_employee_profile(first_key)
print(f"\nProfile saved to: {zip_file}")

In [ ]:
# ── Verify ZIP contents ───────────────────────────────────────────────────────
with zipfile.ZipFile(zip_file, "r") as zf:
    print("Files inside ZIP:", zf.namelist())
    with zf.open(zf.namelist()[0]) as f:
        reader = csv.DictReader(line.decode() for line in f)
        print("\nPreview (first 2 rows):")
        for i, row in enumerate(reader):
            print(dict(row))
            if i >= 1:
                break

---
## Task 6 – Unzip and Display Data with R

The cell below uses the `%%R` magic (provided by **rpy2**). Install it once with:

```bash
pip install rpy2
```

Then restart the kernel and run the cell.

In [ ]:
# Load rpy2 extension (run once per session)
%load_ext rpy2.ipython

In [ ]:
%%R
# ── R Task: Unzip the Employee Profile folder and display the CSV data ─────────

library(utils)   # unzip()
library(readr)   # read_csv()

# ── Locate the ZIP file ────────────────────────────────────────────────────────
zip_files <- list.files(
  path       = "Employee Profile",
  pattern    = "\\.zip$",
  full.names = TRUE
)

if (length(zip_files) == 0) {
  stop("No ZIP file found in 'Employee Profile/'. Run Task 5 first.")
}

target_zip <- zip_files[1]
cat("Found ZIP:", target_zip, "\n")

# ── Unzip to a temporary directory ────────────────────────────────────────────
extract_dir <- file.path("Employee Profile", "extracted")
dir.create(extract_dir, showWarnings = FALSE, recursive = TRUE)

unzip(target_zip, exdir = extract_dir)
cat("Extracted to:", extract_dir, "\n")

# ── Read the extracted CSV ─────────────────────────────────────────────────────
csv_files <- list.files(extract_dir, pattern = "\\.csv$", full.names = TRUE)
if (length(csv_files) == 0) stop("No CSV found after unzipping.")

emp_data <- read_csv(csv_files[1], show_col_types = FALSE)

# ── Display the data ──────────────────────────────────────────────────────────
cat("\n===== Employee Profile Data =====", "\n")
cat("Rows:", nrow(emp_data), " | Columns:", ncol(emp_data), "\n\n")
print(as.data.frame(emp_data))

# ── Summary statistics ─────────────────────────────────────────────────────────
cat("\n===== Summary Statistics =====\n")
print(summary(emp_data[, sapply(emp_data, is.numeric)]))

---
## Bonus – Quick Exploratory Analysis

In [ ]:
import matplotlib.pyplot as plt

# Average TotalPayBenefits per year
yearly_avg = df.groupby("Year")["TotalPayBenefits"].mean().reset_index()

plt.figure(figsize=(9, 4))
plt.bar(yearly_avg["Year"].astype(str), yearly_avg["TotalPayBenefits"], color="steelblue")
plt.title("Average Total Pay + Benefits per Year (SF Employees)")
plt.xlabel("Year")
plt.ylabel("Avg Total Pay + Benefits ($)")
plt.tight_layout()
plt.show()
print(yearly_avg.to_string(index=False))

In [ ]:
# Top 10 highest-paid employees (by average TotalPayBenefits across all years)
top10 = (
    df.groupby("EmployeeName")["TotalPayBenefits"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top10.columns = ["EmployeeName", "Avg_TotalPayBenefits"]
top10["Avg_TotalPayBenefits"] = top10["Avg_TotalPayBenefits"].round(2)
print("Top 10 Highest-Paid Employees (avg across all years):")
top10